In this notebook, we create the baseline regression model relying on the spike counts of motor units as features.

The notebook with the current choice of parameters generates the following files:
- Intermediate files in *regression_data/S2/mu_regression* folder:
    - force_df_Trap_0_15_1_3_20_intra_hold_False.pkl
    - mu_df_sorted_Trap_0_15_1_3_20_intra_hold_False.pkl
    - mu_df_Trap_0_15_1_3_20_intra_hold_False.pkl
    - mu_summary_df_Trap_0_15_1_3_20_intra_hold_False.pkl

- Results files in *results/S2/fp32* folder:
    - metrics_df_intra_ws0.08_ol50_-1_15_cv_on_mu_nfing_1_2_3_4_5_shuffleseed_10_linear_hold_False.pkl
    - trained_models_intra_ws0.08_ol50_-1_15_cv_on_mu_nfing_1_2_3_4_5_shuffleseed_10_linear_hold_False.pkl
    - y_df_intra_ws0.08_ol50_-1_15_cv_on_mu_nfing_1_2_3_4_5_shuffleseed_10_linear_hold_False.pkl
    - y_dict_intra_ws0.08_ol50_-1_15_cv_on_mu_nfing_1_2_3_4_5_shuffleseed_10_linear_hold_False.pkl

In [ ]:
import logging
import os
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib import gridspec
from matplotlib.patches import Patch
notebook_dir = os.getcwd()
project_dir = os.path.dirname(os.path.dirname(notebook_dir))

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
print(f'Notebook dir: {notebook_dir}\nProject dir: {project_dir}')

from force_regression.config.dataconfig import DataConfig
import force_regression.utils.functions as fn
from force_regression.models.linear_regression import ConventionalRegressionOnMu
from configs.constants import *
logging.getLogger().setLevel(logging.INFO)
%load_ext autoreload
%autoreload 2

In [ ]:
config_path = os.path.join(project_dir, 'configs/config.json')

try:
    with open(config_path, 'r') as config_file:
        config = json.load(config_file)
        root_dir = config['root_dir']
        root_results_dir = config['root_results_dir']
        subject_mappings = config['subject_mappings']
        print(f'Root directory from config: {root_dir}')
except FileNotFoundError:
    print(f"Error: 'config.json' not found in {config_path}")
except json.JSONDecodeError:
    print("Error: 'config.json' is not a valid JSON file.")
except KeyError:
    print("Error: 'root_dir' not found in 'config.json'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Set notebook parameters

This is parameter cell used to run the notebook with different parameters using Papermill


In [ ]:
# UPDATE THESE PARAMETERS BEFORE RUNNING THE NOTEBOOK
subj = "S2"  # subject ID: S1 | S2
sign_mvc = -1   # movement direction: -1 (extension) | 1 (flexion)
# if True load pre-generated regression dataframes from file, else generate it
load_regression_data_from_file = True

profile = False  # if True, profile the model during prediction time
# KEEP THESE PARAMETERS UNCHANGED
mvc = 15     # MVC value used during the experiment
emg_type = 'intra' # emg_type is intramuscular EMG
model_type = 'linear' # linear regression model
overlap_in_perc = 50    # percentage overlap between the analysis windows
win_size_in_sec = 0.08  # window size in seconds
select_dir = True # parameter to select direction of force to be used for regression. If False, both directions are used.
load_multi = [mvc] # [mvc] the default is set to the given MVC value. Another option is a list of MVCs
sweep_win_size = False # flag to sweep on the window size. If True place results in a subfolder
segment_hold = False # if True, segment the hold phase of the force profile, else consider the full force profile
post_process = True  # if True, smooth the predicted force using a low-pass filter
shuffle_fingers_seed = 10  # seed used to shuffle the finger orders prior to cross-validation

# Create data configuration instance

In [ ]:
config = DataConfig(root_dir=root_dir,
                    root_results_dir=root_results_dir,
                    subject=fn.reverse_remap(subj, subject_mappings),
                    task_type="Trap", 
                    finger_type="Individual", 
                    day="Day 1",
                    mvc=mvc, 
                    emg_type=emg_type, 
                    f_samp=10240,
                    subj_map=subject_mappings,
                    segment_hold=segment_hold,
                    verbose=True,
                    common_only=True,
                    images=False,
                    time_to_cut=1,
                    load_multi=load_multi,
                    convreg_temp_data_dir='regression_data',
                    figs_dir='figures')
config.sign_mvc = sign_mvc

In [ ]:
config.load_multi, config.mvc, config.direction, config.emg_type, config.sign_mvc, config.segment_hold


In [ ]:
sign_mvc, emg_type , config.emg_type, config.sign_mvc, mvc

# Linear Regression with Motor Units

In [ ]:
mu_reg = ConventionalRegressionOnMu(model_type,
                                    config,
                                    emg_type,
                                    regression_data_parent_dir=config.convreg_temp_data_path,
                                    overlap_in_perc=overlap_in_perc,
                                    window_size_in_sec=win_size_in_sec,
                                    post_process=post_process,
                                    load_regression_data_from_file=load_regression_data_from_file,
                                    shuffle_fingers_seed=shuffle_fingers_seed
                                    )

In [ ]:
mu_reg.create_mu_regression_df()

In [ ]:
# Print number of windows after binning spike counts for each repetition and finger
mu_reg.windows_count_for_finger, mu_reg.data_config.direction


In [ ]:
metrics_df_cv, y_df_cv, trained_models_df = mu_reg.cross_validate_model()

In [ ]:
if profile:
    # Testing the profile_model function
    profiling_results = mu_reg.profile_model(trained_models_df,
                         mu_reg.reg_data_df,
                         non_mus_cols=mu_reg.data_config.force_cols_list + mu_reg.force_aux_cols,
                         verbose=True)
    mu_reg.save_profiling_results(profiling_results)

In [ ]:
metrics_df_cv.groupby([FOLD_COL, FING_ID_COL])['R2_test_post'].mean()

In [ ]:
# sanity check: dimensions of predicted force (total_n_windows, n_fingers)
print(f"shape of predictions for fold 0: {y_df_cv.iloc[0][Y_TRUE_TRAIN].shape}, fold 1: {y_df_cv.iloc[1][Y_TRUE_TRAIN].shape}")

In [ ]:
postprocessed_y_dict = mu_reg.post_process_y_cv_df(y_df_cv)

In [ ]:
"""
A quick debugging plot: visualize the true force profiles on all fingers for each target finger task and fold.
(2 repetitions in total).
Active finger should have higher force levels than the rest reach a plateau at 15% MVC.
The plot shows the order of the fingers in the regression dataframe.
"""
n_folds = len(y_df_cv)
fig = plt.figure(figsize=(10,4))
gs = gridspec.GridSpec(nrows=1, ncols=n_folds)
buffer_time = 2  # in seconds

for fold_i in range(n_folds):
    ax = fig.add_subplot(gs[fold_i])
    fing_order = np.array(y_df_cv[FING_ORDER_COL].iloc[fold_i])
    print(fing_order)
    ax.set_prop_cycle(color=config.finger_color_list)
    for y in [Y_TRUE_TEST]:
        time_ax_shift = 0
        for fing in fing_order:
            plot_alpha = 0.5 if 'true' not in y else 0.8
            y_df = postprocessed_y_dict[y]
            y_df_fold = y_df[y_df[FOLD_COL]==fold_i]
            temp_df = y_df_fold[y_df_fold[FING_ID_COL]==fing]
            time_ax = temp_df[TIME] + time_ax_shift
            ax.plot(time_ax, temp_df[np.arange(config.n_ind_fingers)],  alpha=0.5)
            time_ax_shift = time_ax.iloc[-1] + buffer_time
    ax.set_title(f'Fold {fold_i+1}')
    ax.set_ylabel(FORCE_MVC_LABEL)
    ax.set_xlabel('Time (s)')
    sns.despine(ax=ax, trim=False)
handles = [Patch(facecolor=c, edgecolor=config.color_dict['midnight_blue'], alpha=LEGEND_ALPHA) for  c in config.finger_color_list]
labels = [fing_name for fing_name in config.finger_label_map.keys()]
fig.legend(handles, labels, loc='center right', title='Fingers', bbox_to_anchor=(1.1, 0.5))
fig.text(0.3, 1, 
         f'Overlap: {overlap_in_perc}%, Win size {win_size_in_sec} s. |{config.direction}|{emg_type}|sign_mvc: {config.sign_mvc}')
fig.tight_layout()

In [ ]:
logging.info(f"Saving results to {mu_reg.data_config.results_path}")
mu_reg.save_results(metrics_df_cv, y_df_cv, postprocessed_y_dict, trained_models_df,
                    is_sweep_experiment=sweep_win_size)